In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
cpu = tf.config.list_physical_devices("CPU")[0]
tf.config.set_visible_devices(cpu)

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
path_to_model = "./saving/me_int/SavedModel/"
path_to_data = "./../datagen/data/test/"
path_to_figs = "./postproc/testing/"

In [ ]:
# Model
model = tf.keras.models.load_model(path_to_model)
# Model - Forward reaction rate
model_fe = tf.keras.Model(
  inputs=model.inputs,
  outputs=model.layers[5].output
)
model_fd = tf.keras.Model(
  inputs=model.inputs,
  outputs=model.layers[6].output
)

In [ ]:
I, J = np.tril_indices(3, k=-1)

def get_data(filename):
  # Collect data
  data = pd.read_csv(filename).to_numpy()
  *x, y = np.split(data, np.cumsum([5, 3, 9]), axis=1)
  t = x[0][:,:1].squeeze()
  # Make prediction
  y_pred = model(x).numpy().squeeze()
  fe = tf.math.exp(model_fe(x)).numpy().squeeze()[:,I,J]
  fd = tf.math.exp(model_fd(x)).numpy().squeeze()
  return t, y, y_pred, fe, fd

In [ ]:
def plot_data(
  x=None,
  y=None,
  xref=None,
  yref=None,
  labels=[r'$t$ [s]', r'$X$'],
  scales=['log', 'log'],
  names=None,
  figname=None
):
  # Initialize figures
  fig = plt.figure()
  ax = fig.add_subplot()
  # x axis
  xlim = [np.amin(x), np.amax(x)]
  ax.set_xlim(xlim)
  ax.set_xlabel(labels[0])
  ax.set_xscale(scales[0])
  # y axis
  ax.set_ylabel(labels[1])
  ax.set_yscale(scales[1])
  # Plotting
  n, d = y.shape if (x is not None) else yref.shape
  true_style = dict(
    lw=1
  )
  pred_style = dict(
    marker='^',
    fillstyle='none',
    markersize=5,
    markevery=n//20,
    ls='--',
    lw=1
  )
  for i in range(d):
    if (x is not None):
      ax.plot(x, y[:,i], c=COLORS[i], **pred_style)
    if (xref is not None):
      ax.plot(xref, yref[:,i], c=COLORS[i], **true_style)
    ax.plot([], [], c=COLORS[i], label=names[i])
  plt.legend()
  if (figname is not None):
    fig.savefig(figname)
  plt.close()

In [ ]:
for (i, case) in enumerate(os.listdir(path_to_data)):
  filename = path_to_data + "/" + case
  t, y, y_pred, fe, fd = get_data(filename)
  path_to_figsi = path_to_figs + "/" + case.split(".")[0]
  os.makedirs(path_to_figsi, exist_ok=True)
  plot_data(
    x=t[1:],
    y=fe[1:],
    labels=[r'$t$ [s]', r'$f$'],
    names=[r'$f_{%d%d}$' % (I[i]+1,J[i]+1) for i in range(3)],
    scales=['log', 'log'],
    figname=path_to_figsi + f"/fe.png"
  )
  plot_data(
    x=t[1:],
    y=fd[1:],
    labels=[r'$t$ [s]', r'$f$'],
    names=[r'$f_{%d o}$' % (i+1) for i in range(3)],
    scales=['log', 'log'],
    figname=path_to_figsi + f"/fd.png"
  )